# 🛍 Online Retail Data Analysis

This notebook contains the full workflow for analyzing the Online Retail dataset to understand sales trends and customer behavior.

## 1. 📦 Pull the Data

In [173]:
import pandas as pd

df = pd.read_excel("../data/online-retail-dataset.xlsx")

## 2. 🔎 Peek the Data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
null_info = df.isnull().sum().reset_index()
null_info.columns = ['Column', 'Null Count']
null_info = null_info[null_info['Null Count'] > 0]
null_info

In [ ]:
duplicates = df[df.duplicated()].copy()
cols_with_nulls = ['Description', 'CustomerID']
merge_columns = [col for col in df.columns if col not in cols_with_nulls]
duplicate_counts = (
    duplicates.groupby(merge_columns)
    .size()
    .reset_index(name='DuplicateCount')
)
duplicates = duplicates.merge(duplicate_counts, on=merge_columns, how='left')
duplicates = duplicates.sort_values(by='DuplicateCount', ascending=False).reset_index(drop=True)
duplicates

## 2. 🧹 Data Cleaning

In [ ]:
# drop duplicates
cdf = df.drop_duplicates().copy()
cdf

In [ ]:
# convert dtypes
cdf['InvoiceNo'] = cdf['InvoiceNo'].astype('string')
cdf['StockCode'] = cdf['StockCode'].astype('string')
cdf['Description'] = cdf['Description'].astype('string')
cdf['Country'] = cdf['Country'].astype('string')
cdf['InvoiceDate'] = pd.to_datetime(cdf['InvoiceDate'])
cdf['CustomerID'] = pd.to_numeric(cdf['CustomerID'], errors='coerce').astype('Int64')
cdf.info()

In [ ]:
cdf['StockCode'] = cdf['StockCode'].str.upper()
cdf['Original_Description'] = cdf['Description']
cdf_with_null_desc = cdf.copy()
cdf_with_null_desc['Description'] = cdf_with_null_desc['Description'].fillna('NaN')
grouped = (
    cdf_with_null_desc.groupby(['StockCode', 'Description'])
    .size()
    .reset_index(name='Count')
)
stockcodes = {}
for _, row in grouped.iterrows():
    stockcode = row['StockCode']
    description = row['Description']
    count = row['Count']

    if stockcode not in stockcodes:
        stockcodes[stockcode] = {}

    stockcodes[stockcode][description] = count
most_common_desc = {
    code: max(descs.items(), key=lambda x: x[1])[0]
    for code, descs in stockcodes.items()
}
cdf['Standardized_Description'] = cdf['StockCode'].map(most_common_desc).astype('string')
# cdf['Standardized_Description'] = cdf['Standardized_Description'].astype('string')
cdf['Standardized_Description'] = cdf['Standardized_Description'].replace('NaN', pd.NA)
cdf['Description'] = cdf['Standardized_Description'].fillna(cdf['Original_Description'])
cdf.drop(columns=['Standardized_Description'], inplace=True)
cdf

In [ ]:
cdf.info()

In [ ]:
# show txns with populated desc
filled_desc = cdf[cdf['Description'].notna() & cdf['Original_Description'].isna()].copy()
filled_desc

In [ ]:
# see updated nulls
null_info = cdf.isnull().sum().reset_index()
null_info.columns = ['Column', 'Null Count']
null_info = null_info[null_info['Null Count'] > 0]
null_info

In [ ]:
# Add TotalSale Column
cdf['TotalSale'] = cdf['Quantity'] * cdf['UnitPrice']
cdf

## 3. 📊 Exploratory Data Analysis (EDA)

In [185]:
cdf_purchases = cdf[cdf['Quantity'] > 0].copy()

In [186]:
cdf_returns = cdf[cdf['Quantity'] < 0].copy()

In [187]:
gross_sales = cdf_purchases['TotalSale'].sum()

In [188]:
total_returns = -cdf_returns['TotalSale'].sum()

In [189]:
net_sales = cdf['TotalSale'].sum()

In [190]:
aov = cdf_purchases.groupby('InvoiceNo')['TotalSale'].sum().mean()

In [191]:
print("Gross Sales: £{:.2f}".format(gross_sales))
print("Total Returns: £{:.2f}".format(total_returns))
print("Net Sales: £{:.2f}".format(net_sales))
print("Average Order Value: £{:.2f}".format(aov))

Gross Sales: £10619986.68
Total Returns: £893979.73
Net Sales: £9726006.95
Average Order Value: £512.35


## 4. 📈 Visualizations

In [ ]:
# Add charts

## 5. 💡 Insights and Recommendations

- Insight 1: _<Write here>_
- Insight 2: _<Write here>_
- Insight 3: _<Write here>_

**Recommendations:**
- Recommendation 1: _<Write here>_
- Recommendation 2: _<Write here>_
- Recommendation 3: _<Write here>_